In [ ]:
import pickle
import json
import os
from shapely.geometry import mapping

def pickle_to_geojson(pickle_path, output_path):
    """
    将pickle格式的城市规划数据转换为GeoJSON格式
    
    参数:
    pickle_path: 输入的pickle文件路径
    output_path: 输出的GeoJSON文件路径
    """
    # 加载pickle数据
    print(f"正在读取 {pickle_path}...")
    with open(pickle_path, 'rb') as f:
        plan_data = pickle.load(f)
    
    # 打印数据结构信息，帮助理解
    print(f"数据结构类型: {type(plan_data)}")
    
    # 如果是GeoDataFrame，可以直接转换为GeoJSON
    if hasattr(plan_data, 'to_crs') and hasattr(plan_data, 'to_json'):
        print("检测到GeoDataFrame，直接转换为GeoJSON...")
        # 确保使用WGS84坐标系统（GeoJSON标准）
        plan_data = plan_data.to_crs("EPSG:4326")
        geojson_data = plan_data.to_json()
        
        # 保存为GeoJSON文件
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(geojson_data)
        print(f"已保存GeoJSON到 {output_path}")
        return

    # 如果不是GeoDataFrame，需要手动构建GeoJSON
    print("构建GeoJSON格式...")
    
    # 创建GeoJSON结构
    geojson = {
        "type": "FeatureCollection",
        "features": []
    }
    
    # 根据plan_data的结构构建features
    # 这部分需要根据实际数据结构调整
    if isinstance(plan_data, dict):
        for key, value in plan_data.items():
            if hasattr(value, 'geometry'):
                # 如果有geometry属性，转换为Feature
                feature = {
                    "type": "Feature",
                    "geometry": mapping(value.geometry),
                    "properties": {
                        "id": key
                    }
                }
                # 添加其他属性
                for attr in value.__dict__:
                    if attr != 'geometry' and not attr.startswith('_'):
                        feature["properties"][attr] = getattr(value, attr)
                
                geojson["features"].append(feature)
    
    # 保存为GeoJSON文件
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(geojson, f, indent=2)
    
    print(f"已保存GeoJSON到 {output_path}")


pickle_path = "init_plan_g6.pickle"  # 输入文件路径
output_path = "init_plan_g6.geojson"  # 输出文件路径
    
pickle_to_geojson(pickle_path, output_path)

In [5]:
import geojson

features = []

# 区域尺寸
length = 3000  # 米
width = 3000   # 米
grid_size = 500  # 米

id_counter = 1

# 生成小正方形的顶点和边
for x in range(0, length, grid_size):
    for y in range(0, width, grid_size):
        # 定义四个顶点坐标
        v1 = [x, y]
        v2 = [x + grid_size, y]
        v3 = [x + grid_size, y + grid_size]
        v4 = [x, y + grid_size]

        # 创建顶点要素，类型为 13
        for vx, vy in [v1, v2, v3, v4]:
            point = geojson.Point([vx, vy])
            feature = geojson.Feature(
                geometry=point,
                properties={
                    "id": id_counter,
                    "type": 13,
                    "existence": True
                }
            )
            features.append(feature)
            id_counter += 1

        # 创建边要素，类型为 2
        edges = [
            [v1, v2],
            [v2, v3],
            [v3, v4],
            [v4, v1]
        ]
        for edge in edges:
            line = geojson.LineString(edge)
            feature = geojson.Feature(
                geometry=line,
                properties={
                    "id": id_counter,
                    "type": 2,
                    "existence": True
                }
            )
            features.append(feature)
            id_counter += 1

        # 可选：创建小正方形的多边形要素（如果需要）
        polygon = geojson.Polygon([[v1, v2, v3, v4, v1]])
        feature = geojson.Feature(
            geometry=polygon,
            properties={
                "id": id_counter,
                "type": 1,  # 多边形类型，可根据需要调整
                "existence": True
            }
        )
        features.append(feature)
        id_counter += 1

# 创建 FeatureCollection
feature_collection = geojson.FeatureCollection(features)

# 保存为 geojson 文件
with open('g6.geojson', 'w') as f:
    geojson.dump(feature_collection, f)

In [6]:
import geopandas as gpd
import matplotlib.pyplot as plt

# 加载 GeoJSON 文件
gdf = gpd.read_file('g6.geojson')

# 查看前几行数据
print(gdf.head())

   id  type  existence                                         geometry
0   1    13       True                          POINT (0.00000 0.00000)
1   2    13       True                        POINT (500.00000 0.00000)
2   3    13       True                      POINT (500.00000 500.00000)
3   4    13       True                        POINT (0.00000 500.00000)
4   5     2       True  LINESTRING (0.00000 0.00000, 500.00000 0.00000)
